# 01. Сбор данных для проекта

**ЛР 2, шаг 6.** Ноутбук загружает исходные данные для проекта по доступности школ и поликлиник в Уфе.

Из OpenStreetMap загружаются граница Уфы, административные районы (если они размечены в OSM), школы и медицинские учреждения. Отдельно используется CSV «Список поликлиник» с портала открытых данных Республики Башкортостан. CSV нужно предварительно скачать и положить в `data/raw/clinics_source.csv`.

Исходные данные сохраняются в `data/raw/`. Эта папка должна быть исключена из Git.

Данные OpenStreetMap распространяются по лицензии ODbL 1.0. Указывайте **«© участники OpenStreetMap»** и ссылку https://www.openstreetmap.org/copyright.

[OpenStreetMap](https://www.openstreetmap.org/) · [Overpass Turbo](https://overpass-turbo.eu) · [Список поликлиник РБ](https://opendata.sf2.simai.ru/datasets/health/30/)

## 1. Подготовка среды

Ноутбук можно запускать в Google Colab или локально в Jupyter / VS Code.

В Google Colab укажите имя общей папки команды в `PROJECT_FOLDER`, запустите ячейки по порядку и разрешите подключение Google Диска. При локальном запуске ноутбук предполагает, что он открыт из папки `notebooks/` репозитория. Данные появятся в `data/raw/`.

In [ ]:
import sys
import subprocess
from pathlib import Path

PROJECT_FOLDER = "pm-lab-2-schoolmed-access"  # папка команды в Google Drive
IN_COLAB = "google.colab" in sys.modules

packages = ["geopandas>=1.0", "osmnx>=2.0", "pandas", "folium", "mapclassify"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    if not BASE.exists():
        raise FileNotFoundError(f"Не найдена папка {BASE}. Проверьте PROJECT_FOLDER.")
else:
    BASE = Path.cwd().parent  # запуск из notebooks/

RAW = BASE / "data" / "raw"
MANUAL = BASE / "data" / "manual"
PROCESSED = BASE / "data" / "processed"
for folder in (RAW, MANUAL, PROCESSED):
    folder.mkdir(parents=True, exist_ok=True)

print("Среда:", "Google Colab" if IN_COLAB else "локальная")
print("Корень проекта:", BASE.resolve())
print("Папка исходных данных:", RAW.resolve())

## 2. Параметры загрузки

Объект появится в выгрузке OSM, только если он нанесён на OpenStreetMap и размечен соответствующим тегом. Список поликлиник из регионального CSV является отдельным источником и не заменяется слоем медицинских учреждений OSM.

In [ ]:
from datetime import datetime, timezone
import json
import warnings

import geopandas as gpd
import pandas as pd
import osmnx as ox

PLACE = "Ufa, Russia"
DISTRICT_ADMIN_LEVEL = "9"  # проверьте, что найденные полигоны действительно являются районами Уфы

LAYERS = {
    "schools": {"amenity": "school"},
    "clinics_osm": {
        "amenity": ["clinic", "doctors"],
        "healthcare": ["clinic", "doctor"]
    }
}

KEEP_COLUMNS = [
    "name", "amenity", "healthcare", "operator",
    "addr:street", "addr:housenumber", "addr:city",
    "opening_hours", "website"
]

ox.settings.use_cache = True
ox.settings.cache_folder = str(RAW / "cache")
ox.settings.requests_timeout = 300
ox.settings.log_console = False

print("Территория:", PLACE)
print("Слои:", list(LAYERS))

## 3. Вспомогательная функция и журнал загрузки

Пространственные слои сохраняются в GeoPackage (`.gpkg`) с системой координат EPSG:4326 (WGS 84).

In [ ]:
def to_layer(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Оставляет идентификатор OSM, выбранные атрибуты и геометрию."""
    gdf = gdf.reset_index()
    gdf = gdf.rename(columns={
        "element": "osm_type", "element_type": "osm_type",
        "id": "osm_id", "osmid": "osm_id"
    })
    if "osm_type" not in gdf.columns:
        gdf["osm_type"] = None
    if "osm_id" not in gdf.columns:
        gdf["osm_id"] = None

    columns = ["osm_type", "osm_id"] + [
        c for c in KEEP_COLUMNS if c in gdf.columns
    ] + ["geometry"]
    gdf = gdf[columns].copy()

    for c in gdf.columns:
        if c != "geometry" and gdf[c].dtype == object:
            gdf[c] = gdf[c].map(
                lambda v: v if v is None or isinstance(v, str) else str(v)
            )

    return gdf.set_crs("EPSG:4326", allow_override=True)

download_log = {
    "place": PLACE,
    "downloaded_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source": "OpenStreetMap via Overpass API (OSMnx)",
    "license": "ODbL 1.0",
    "crs": "EPSG:4326",
    "layers": {}
}

## 4. Граница Уфы и административные районы

Районы могут отсутствовать или иметь другую разметку в OSM. Если слой районов пустой, это нужно зафиксировать и подобрать дополнительный официальный источник.

In [ ]:
boundary = ox.geocode_to_gdf(PLACE)
if boundary.empty:
    raise RuntimeError(f"Не удалось получить границу для {PLACE}")

boundary = boundary[["display_name", "geometry"]].copy()
boundary = boundary.to_crs("EPSG:4326")
boundary_path = RAW / "boundary.gpkg"
boundary.to_file(boundary_path, driver="GPKG")
download_log["layers"]["boundary"] = {
    "file": boundary_path.name, "features": int(len(boundary))
}
print("Граница территории:", len(boundary))

admin = ox.features_from_place(PLACE, tags={"boundary": "administrative"})
if "admin_level" in admin.columns:
    admin = admin[
        (admin["admin_level"].astype(str) == DISTRICT_ADMIN_LEVEL)
        & admin.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    ].copy()
    districts = to_layer(admin)
else:
    districts = gpd.GeoDataFrame(
        columns=["osm_type", "osm_id", "geometry"],
        geometry="geometry", crs="EPSG:4326"
    )

districts_path = RAW / "districts.gpkg"
if not districts.empty:
    districts.to_file(districts_path, driver="GPKG")
    download_log["layers"]["districts"] = {
        "file": districts_path.name, "features": int(len(districts))
    }
    print("Административные полигоны:", len(districts))
    if "name" in districts.columns:
        display(districts[["name"]].head(30))
else:
    download_log["layers"]["districts"] = {
        "file": None, "features": 0,
        "note": "Не найдены полигоны с указанным admin_level"
    }
    print("Районы не найдены. Проверьте admin_level или используйте отдельный источник.")

## 5. Загрузка школ и медицинских учреждений из OpenStreetMap

Загрузка может занять несколько минут. При ошибке 429 или 504 от Overpass API подождите и повторите ячейку. OSMnx использует локальный кэш запросов.

In [ ]:
for layer_name, tags in LAYERS.items():
    print(f"Загружается слой {layer_name}...")
    raw_gdf = ox.features_from_place(PLACE, tags=tags)
    layer_gdf = to_layer(raw_gdf)
    output_path = RAW / f"{layer_name}.gpkg"

    if layer_gdf.empty:
        warnings.warn(f"Слой {layer_name} пуст. Файл не записан.")
        download_log["layers"][layer_name] = {
            "file": None, "features": 0, "tags": tags
        }
        continue

    layer_gdf.to_file(output_path, driver="GPKG")
    download_log["layers"][layer_name] = {
        "file": output_path.name,
        "features": int(len(layer_gdf)),
        "geometry_types": {
            str(k): int(v) for k, v in layer_gdf.geom_type.value_counts().items()
        },
        "tags": tags
    }
    print(layer_name, len(layer_gdf), layer_gdf.geom_type.value_counts().to_dict())

## 6. Загрузка списка поликлиник Республики Башкортостан

Источник: [«Список поликлиник»](https://opendata.sf2.simai.ru/datasets/health/30/).

Скачайте CSV со страницы набора и положите его в `data/raw/clinics_source.csv`. Ноутбук читает UTF-8 или CP1251. Если столбец муниципалитета не будет определён автоматически, укажите его точное название в `MUNICIPALITY_COLUMN`.

In [ ]:
CLINICS_CSV = RAW / "clinics_source.csv"
if not CLINICS_CSV.exists():
    raise FileNotFoundError(
        "Не найден data/raw/clinics_source.csv. Скачайте CSV со страницы "
        "https://opendata.sf2.simai.ru/datasets/health/30/ "
        "и сохраните его под именем clinics_source.csv."
    )

try:
    clinics_source = pd.read_csv(CLINICS_CSV, encoding="utf-8")
    clinics_encoding = "utf-8"
except UnicodeDecodeError:
    clinics_source = pd.read_csv(CLINICS_CSV, encoding="cp1251")
    clinics_encoding = "cp1251"

print("Кодировка:", clinics_encoding)
print("Столбцы:", clinics_source.columns.tolist())
display(clinics_source.head())

# Если автоопределение не сработает, впишите точное имя столбца.
MUNICIPALITY_COLUMN = None
candidate_names = [
    "Муниципальное образование", "Муниципальный район",
    "Муниципалитет", "Municipality", "municipality"
]
normalized = {str(c).strip().casefold(): c for c in clinics_source.columns}
if MUNICIPALITY_COLUMN is None:
    for candidate in candidate_names:
        if candidate.casefold() in normalized:
            MUNICIPALITY_COLUMN = normalized[candidate.casefold()]
            break

if MUNICIPALITY_COLUMN is None:
    raise KeyError(
        "Не найден столбец муниципалитета. Посмотрите список столбцов выше "
        "и укажите его название в MUNICIPALITY_COLUMN."
    )

clinics_ufa = clinics_source[
    clinics_source[MUNICIPALITY_COLUMN].astype(str).str.contains(
        "Уфа", case=False, na=False
    )
].copy()

print("Записей для Уфы:", len(clinics_ufa))
clinics_ufa.to_csv(RAW / "clinics_ufa.csv", index=False, encoding="utf-8-sig")

download_log["regional_clinics_csv"] = {
    "source": "https://opendata.sf2.simai.ru/datasets/health/30/",
    "input_file": CLINICS_CSV.name,
    "filtered_file": "clinics_ufa.csv",
    "encoding": clinics_encoding,
    "municipality_column": str(MUNICIPALITY_COLUMN),
    "features": int(len(clinics_ufa))
}
display(clinics_ufa.head(20))

## 7. Проверка пространственных данных

Это первичная проверка чтения файлов, CRS и пустых геометрий. Она не заменяет полноценный контроль качества и проверку соответствия объектов территории Уфы.

In [ ]:
for filename in [
    "boundary.gpkg", "districts.gpkg", "schools.gpkg", "clinics_osm.gpkg"
]:
    path = RAW / filename
    if not path.exists():
        print(filename, ": файл отсутствует")
        continue

    gdf = gpd.read_file(path)
    empty_geom = int(gdf.geometry.is_empty.sum()) if len(gdf) else 0
    null_geom = int(gdf.geometry.isna().sum()) if len(gdf) else 0
    print(
        f"{filename}: объектов={len(gdf)}, CRS={gdf.crs}, "
        f"пустых геометрий={empty_geom}, пропущенных={null_geom}"
    )

print("CSV поликлиник Уфы:", len(clinics_ufa))

## 8. Быстрая визуальная проверка

На карте отображаются граница Уфы, школы и медицинские объекты из OSM. CSV поликлиник пока не отображается: в нём адреса, а не координаты. Геокодирование адресов выполняется на следующем этапе.

In [ ]:
map_view = boundary.explore(
    color="gray", style_kwds={"fillOpacity": 0.05},
    tooltip=["display_name"]
)

schools_path = RAW / "schools.gpkg"
if schools_path.exists():
    schools_map = gpd.read_file(schools_path)
    if not schools_map.empty:
        tips = [c for c in ["name", "amenity"] if c in schools_map.columns]
        map_view = schools_map.explore(m=map_view, color="blue", tooltip=tips)

clinics_path = RAW / "clinics_osm.gpkg"
if clinics_path.exists():
    clinics_map = gpd.read_file(clinics_path)
    if not clinics_map.empty:
        tips = [c for c in ["name", "amenity", "healthcare"] if c in clinics_map.columns]
        map_view = clinics_map.explore(m=map_view, color="red", tooltip=tips)

map_view

## 9. Сохранение журнала загрузки

Дата в журнале – дата выполнения ноутбука, а не дата последнего обновления данных на сайте.

In [ ]:
download_log["completed_at_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
log_path = RAW / "_download_log.json"
log_path.write_text(
    json.dumps(download_log, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("Журнал сохранён:", log_path)
print(json.dumps(download_log, ensure_ascii=False, indent=2))

## 10. Ожидаемые файлы и воспроизводимость

В `data/raw/` ожидаются:

| Файл | Содержимое |
|---|---|
| `boundary.gpkg` | Граница Уфы |
| `districts.gpkg` | Районы Уфы из OSM, если найдены |
| `schools.gpkg` | Школы из OSM |
| `clinics_osm.gpkg` | Медицинские объекты из OSM |
| `clinics_source.csv` | Исходный CSV поликлиник |
| `clinics_ufa.csv` | Записи CSV, отфильтрованные по Уфе |
| `_download_log.json` | Журнал загрузки |
| `cache/` | Кэш запросов OSMnx |

Другой участник команды должен получить доступ к общей папке, скачать CSV в `data/raw/clinics_source.csv`, указать `PROJECT_FOLDER` (для Colab), запустить ноутбук сверху вниз и проверить файлы и журнал.

Количество объектов при повторном запуске может отличаться, поскольку источники обновляются. Важно, чтобы повторялась процедура загрузки.

### Если загрузка не работает

- **Ошибка 429 или 504:** подождите несколько минут и повторите запрос.
- **Не найдена граница:** проверьте `PLACE`.
- **Не найдены районы:** проверьте `DISTRICT_ADMIN_LEVEL`; не подменяйте административные районы территориальными зонами.
- **Не найден CSV:** скачайте его со страницы источника и проверьте имя файла.
- **Не найден столбец муниципалитета:** посмотрите названия столбцов и задайте `MUNICIPALITY_COLUMN` вручную.